# Kaggle Playground Series S6E6 — Stellar Classification (GALAXY / QSO / STAR)

**Competition:** [Playground Series - Season 6, Episode 6](https://www.kaggle.com/competitions/playground-series-s6e6)
**Task:** SDSS(Sloan Digital Sky Survey) 관측 데이터를 이용해 천체를 `GALAXY`, `QSO`(퀘이사), `STAR` 세 클래스로 분류
**Metric:** Balanced Accuracy
**Final Public LB Score:** `0.96744` (3-model ensemble: XGBoost + LightGBM + CatBoost)
**Final Private LB Score:** `0.96685` (3-model ensemble: XGBoost + LightGBM + CatBoost)

## Approach 요약
1. 색지수(color index), 좌표 변환, redshift 상호작용 등 도메인 지식 기반 feature engineering
2. StratifiedKFold(5-fold) 기반 교차검증
3. STAR↔GALAXY confusion 완화를 위한 `sample_weight` 조정
4. 모델별(Optuna) 하이퍼파라미터 튜닝 (LightGBM은 로컬 CPU, XGBoost/CatBoost는 Kaggle P100 GPU)
5. 3-model soft-voting 앙상블 + SciPy 기반 클래스별 threshold 보정(Balanced Accuracy 최적화)

> 이 노트북은 재현을 위해 로컬(train.csv/test.csv 필요) 또는 Kaggle 커널에서 셀을 순서대로 실행해야 합니다. 학습에는 5-fold × 3-model 기준 약 20~30분이 소요됩니다.


## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import KBinsDiscretizer
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

sns.set_theme(style="whitegrid")


## 2. Data Loading

`train.csv` / `test.csv`는 [Kaggle 대회 데이터 탭](https://www.kaggle.com/competitions/playground-series-s6e6/data)에서 받아 노트북과 같은 경로에 두면 됩니다.

In [ ]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

print(f"train shape: {train_df.shape}")
print(f"test shape:  {test_df.shape}")
train_df.head()


## 3. Feature Engineering

SDSS 측광(photometric) 데이터의 특성상 각 필터(`u, g, r, i, z`)의 절대값보다 필터 간 차이(색지수, color index)가 천체의 물리적 특성(온도, 스펙트럼 형태)을 훨씬 안정적으로 반영합니다. 아래 파이프라인은 이 도메인 지식을 바탕으로 구성했습니다.

| Feature | 목적 |
|---|---|
| `u-g, g-r, r-i, i-z` (인접 색지수) | 국소적인 스펙트럼 기울기 |
| `u-z, u-r, g-z` (비인접 색지수) | 자외선~적외선 전 구간의 넓은 스펙트럼 경향 (QSO의 UV excess 포착에 유리) |
| `log_redshift` | redshift는 QSO에서 크게 튀는 long-tail 분포라 log1p로 압축, split point를 고르게 함 |
| `coord_x/y/z` | RA/Dec(alpha/delta)를 3D 단위벡터로 변환해 각도 좌표의 순환성(0°=360°) 문제 해결 |
| `redshift_x_uz`, `redshift_x_ug`, `uz_x_ug` | redshift × color 상호작용 — QSO 특유의 "높은 redshift + 특정 색" 조합을 명시적으로 노출 |
| `photo_midrange` | 5개 필터의 (max+min)/2, 전체 밝기 레벨의 robust한 요약 |
| `coord_{x,y,z}_500_bin` | 3D 좌표를 quantile 기준 500구간으로 이산화 (공간적 군집 신호 근사) |

**주의 (data leakage 방지):** `KBinsDiscretizer`는 train에서만 `fit_transform`하고, test에는 저장된 bin 경계로 `transform`만 적용합니다.

In [ ]:
bin_models = {}
filters = ['u', 'g', 'r', 'i', 'z']

def apply_pipeline(df, fit=True):
    df = df.copy()
    df['u-g'] = df['u'] - df['g']
    df['g-r'] = df['g'] - df['r']
    df['r-i'] = df['r'] - df['i']
    df['i-z'] = df['i'] - df['z']
    df['u-z'] = df['u'] - df['z']
    df['u-r'] = df['u'] - df['r']
    df['g-z'] = df['g'] - df['z']
    df['log_redshift'] = np.log1p(np.maximum(0, df['redshift']))
    df['coord_x'] = np.cos(np.radians(df['delta'])) * np.cos(np.radians(df['alpha']))
    df['coord_y'] = np.cos(np.radians(df['delta'])) * np.sin(np.radians(df['alpha']))
    df['coord_z'] = np.sin(np.radians(df['delta']))
    df['redshift_x_uz'] = df['log_redshift'] * df['u-z']
    df['redshift_x_ug'] = df['log_redshift'] * df['u-g']
    df['uz_x_ug'] = df['u-z'] * df['u-g']
    df['photo_midrange'] = (df[filters].max(axis=1) + df[filters].min(axis=1)) / 2

    # bin: train은 fit_transform, test는 transform만
    for col in ['coord_x', 'coord_y', 'coord_z']:
        bin_name = f'{col}_500_bin'
        if fit:
            kb = KBinsDiscretizer(n_bins=500, encode='ordinal', strategy='quantile', subsample=None)
            df[bin_name] = kb.fit_transform(df[[col]]).ravel().astype(int)
            bin_models[bin_name] = kb
        else:
            kb = bin_models[bin_name]  # test용: train에서 생성/저장한 bin 경계 재사용
            df[bin_name] = kb.transform(df[[col]]).ravel().astype(int)

    return df

train = apply_pipeline(train_df, fit=True)
test = apply_pipeline(test_df, fit=False)


## 4. EDA — 클래스별 분포 확인

파생 feature들이 실제로 클래스를 얼마나 잘 구분하는지 시각적으로 점검합니다. 특히 STAR↔GALAXY confusion의 근원으로 지목된 `redshift`, 그리고 색지수 분포를 확인합니다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 클래스 분포
train['class'].value_counts().plot(kind='bar', ax=axes[0], color=['#4C72B0', '#DD8452', '#55A868'])
axes[0].set_title('Class Distribution (train)')
axes[0].set_xlabel('class')
axes[0].set_ylabel('count')

# redshift 분포 (log scale) by class
for cls in train['class'].unique():
    subset = train[train['class'] == cls]
    axes[1].hist(subset['log_redshift'], bins=60, alpha=0.5, label=cls, density=True)
axes[1].set_title('log_redshift Distribution by Class')
axes[1].set_xlabel('log_redshift')
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
color_features = ['u-g', 'g-r', 'r-i', 'i-z', 'u-z']

fig, axes = plt.subplots(1, len(color_features), figsize=(4*len(color_features), 4), sharey=False)
for ax, feat in zip(axes, color_features):
    sns.boxplot(data=train, x='class', y=feat, ax=ax, showfliers=False,
                order=['STAR', 'GALAXY', 'QSO'])
    ax.set_title(feat)

plt.suptitle('Color Index Distribution by Class (outliers hidden)', y=1.03)
plt.tight_layout()
plt.show()


In [ ]:
# STAR/GALAXY confusion 근원 확인: redshift x spectral_type/galaxy_population 조합
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(
    data=train.sample(min(20000, len(train)), random_state=42),
    x='log_redshift', y='u-z', hue='class', alpha=0.4, s=10, ax=ax,
    palette={'STAR': '#55A868', 'GALAXY': '#4C72B0', 'QSO': '#DD8452'}
)
ax.set_title('log_redshift vs u-z, colored by class')
plt.tight_layout()
plt.show()


## 5. Label Encoding

In [ ]:
le = LabelEncoder()
train['target'] = le.fit_transform(train['class'])
print(le.classes_)


## 6. 모델별 Feature Set 구성

- **XGBoost**: `spectral_type`, `galaxy_population`을 One-Hot Encoding
- **LightGBM / CatBoost**: 두 모델 모두 categorical dtype을 네이티브로 지원하므로 그대로 사용

In [ ]:
# For XGBoost (OneHot Encoding)
train_onehot = pd.get_dummies(train, columns=['spectral_type', 'galaxy_population'])
test_onehot = pd.get_dummies(test, columns=['spectral_type', 'galaxy_population'])

features_xgb = ['alpha', 'delta',
                'u', 'g', 'r', 'i', 'z',
                'log_redshift',
                'u-g', 'g-r', 'r-i', 'i-z', 'u-z', 'u-r', 'g-z',
                'spectral_type_A/F', 'spectral_type_G/K',
                'spectral_type_M', 'spectral_type_O/B',
                'galaxy_population_Blue_Cloud',
                'galaxy_population_Red_Sequence',
                'coord_x', 'coord_y', 'coord_z',
                'redshift_x_uz', 'redshift_x_ug', 'uz_x_ug',
                'coord_x_500_bin', 'coord_y_500_bin', 'coord_z_500_bin',
                'photo_midrange']

# For LightGBM / CatBoost (native categorical support, no OneHot needed)
features_lgb = ['alpha', 'delta',
                'u', 'g', 'r', 'i', 'z',
                'log_redshift',
                'u-g', 'g-r', 'r-i', 'i-z', 'u-z', 'u-r', 'g-z',
                'spectral_type', 'galaxy_population',
                'coord_x', 'coord_y', 'coord_z',
                'redshift_x_uz', 'redshift_x_ug', 'uz_x_ug',
                'coord_x_500_bin', 'coord_y_500_bin', 'coord_z_500_bin',
                'photo_midrange']


In [ ]:
X_xgb_full = train_onehot[features_xgb]
xgb_test = test_onehot[features_xgb]

X_lgb_full = train[features_lgb]
lgb_test = test[features_lgb]

X_cat_full = train[features_lgb]

y_full = train['target']


## 7. 모델 학습 — 5-Fold CV, 3-Model Ensemble

### Sample Weight 전략
EDA에서 STAR↔GALAXY confusion이 `spectral_type × galaxy_population` 조합에서 두드러진다는 점을 확인했고, 이를 완화하기 위해 STAR 클래스 중 특정 `galaxy_population` 값을 가지는(즉 혼동되기 쉬운) 샘플에 더 높은 weight를 부여했습니다.
- `STAR & Blue_Cloud` → weight `2.0`
- `STAR & Red_Sequence` → weight `3.5`

### Cross Validation
- `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`
- fold별로 XGBoost / LightGBM / CatBoost 3개 모델을 동일한 sample_weight로 학습
- 각 모델의 하이퍼파라미터는 Optuna로 사전 튜닝 (LightGBM은 로컬 M5 Pro CPU, XGBoost/CatBoost는 Kaggle P100 GPU 환경에서 별도 진행 — GPU 미지원 조합인 LightGBM+Apple Silicon 이슈로 CPU 학습)
- early stopping(50 rounds)으로 과적합 방지

> 아래 셀은 원본 파이프라인과 동일하되, GitHub 가독성을 위해 학습 로그 출력을 최소화했습니다 (LightGBM/XGBoost/CatBoost의 verbose 옵션 조정). 또한 fold별 feature importance를 누적해 이후 시각화에 사용합니다.

In [ ]:
N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

cat_features = ['spectral_type', 'galaxy_population']  # CatBoost cat_features 지정용

# test 예측 저장용
xgb_test_preds = np.zeros((len(xgb_test), 3))
lgb_test_preds = np.zeros((len(lgb_test), 3))
cat_test_preds = np.zeros((len(lgb_test), 3))

# oof 예측 저장용
xgb_oof_preds = np.zeros((len(X_xgb_full), 3))
lgb_oof_preds = np.zeros((len(X_xgb_full), 3))
cat_oof_preds = np.zeros((len(X_xgb_full), 3))

# feature importance 누적용 (분석 셀에서 사용)
lgb_importances = []
xgb_importances = []
cat_importances = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_xgb_full, y_full)):
    print(f"\n{'='*50}")
    print(f"Fold {fold+1}/{N_FOLDS}")
    print(f"{'='*50}")

    # train/val split
    X_tr_xgb, X_val_xgb = X_xgb_full.iloc[train_idx], X_xgb_full.iloc[val_idx]
    X_tr_lgb, X_val_lgb = X_lgb_full.iloc[train_idx], X_lgb_full.iloc[val_idx]
    X_tr_cat, X_val_cat = X_cat_full.iloc[train_idx], X_cat_full.iloc[val_idx]
    y_tr, y_val = y_full.iloc[train_idx], y_full.iloc[val_idx]

    # sample_weight 계산 (3 models 공통)
    sample_weight = np.ones(len(y_tr))
    star_label = list(le.classes_).index('STAR')

    star_blue_mask = ((y_tr == star_label) &
                     (X_tr_lgb['galaxy_population'] == 'Blue_Cloud'))
    sample_weight[star_blue_mask.values] = 2.0

    star_red_mask = ((y_tr == star_label) &
                    (X_tr_lgb['galaxy_population'] == 'Red_Sequence'))
    sample_weight[star_red_mask.values] = 3.5

    # LGB / CatBoost category type
    X_tr_lgb = X_tr_lgb.copy()
    X_val_lgb = X_val_lgb.copy()
    X_tr_cat = X_tr_cat.copy()
    X_val_cat = X_val_cat.copy()
    for col in cat_features:
        X_tr_lgb[col] = X_tr_lgb[col].astype('category')
        X_val_lgb[col] = X_val_lgb[col].astype('category')
        lgb_test[col] = lgb_test[col].astype('category')
        X_tr_cat[col] = X_tr_cat[col].astype(str)
        X_val_cat[col] = X_val_cat[col].astype(str)

    # LightGBM
    lgb_model = lgb.LGBMClassifier(
        n_estimators=10000,
        learning_rate=0.0337844450084976,
        max_depth=4,
        num_leaves=300,
        min_child_samples=62,
        subsample=0.5182304505438041,
        colsample_bytree=0.5976563561780116,
        reg_alpha=3.0067160584908317,
        reg_lambda=0.007775801747599063,
        n_jobs=-1,
        objective='multiclass',
        num_class=3,
        random_state=42,
        verbosity=-1
    )
    lgb_model.fit(X_tr_lgb, y_tr,
                 sample_weight=sample_weight,
                 eval_set=[(X_val_lgb, y_val)],
                 callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])

    # XGBoost
    xgb_model = xgb.XGBClassifier(
        n_estimators=10000,
        learning_rate=0.01,
        max_depth=6,
        min_child_weight=5,
        subsample=0.7774574305987338,
        colsample_bytree=0.6717228980292207,
        reg_alpha=0.001966439125342902,
        reg_lambda=0.00010503049215744636,
        gamma=9.066670037618216e-05,
        n_jobs=-1,
        objective='multi:softmax',
        num_class=3,
        random_state=42,
        eval_metric='mlogloss',
        early_stopping_rounds=50
    )
    xgb_model.fit(X_tr_xgb, y_tr,
                 sample_weight=sample_weight,
                 eval_set=[(X_val_xgb, y_val)],
                 verbose=False)

    # CatBoost
    cat_model = CatBoostClassifier(
        iterations=10000,
        learning_rate=0.1,
        depth=6,
        l2_leaf_reg=7.616892961846623,
        bagging_temperature=0.09373578572264332,
        random_strength=0.009895905889716963,
        border_count=254,
        loss_function='MultiClass',
        random_state=42,
        cat_features=cat_features,
        task_type='CPU',
        verbose=False,
        early_stopping_rounds=50
    )
    cat_model.fit(X_tr_cat, y_tr,
                 sample_weight=sample_weight,
                 eval_set=(X_val_cat, y_val))

    # oof 예측
    xgb_oof_preds[val_idx] = xgb_model.predict_proba(X_val_xgb)
    lgb_oof_preds[val_idx] = lgb_model.predict_proba(X_val_lgb)
    cat_oof_preds[val_idx] = cat_model.predict_proba(X_val_cat)

    # test 예측 누적
    xgb_test_preds += xgb_model.predict_proba(xgb_test) / N_FOLDS
    lgb_test_preds += lgb_model.predict_proba(lgb_test) / N_FOLDS
    cat_test_preds += cat_model.predict_proba(lgb_test) / N_FOLDS

    # feature importance 누적
    lgb_importances.append(lgb_model.feature_importances_)
    xgb_importances.append(xgb_model.feature_importances_)
    cat_importances.append(cat_model.feature_importances_)

    # fold 점수
    fold_xgb = balanced_accuracy_score(y_val, xgb_oof_preds[val_idx].argmax(axis=1))
    fold_lgb = balanced_accuracy_score(y_val, lgb_oof_preds[val_idx].argmax(axis=1))
    fold_cat = balanced_accuracy_score(y_val, cat_oof_preds[val_idx].argmax(axis=1))
    fold_ensemble = balanced_accuracy_score(y_val,
                    ((xgb_oof_preds[val_idx] + lgb_oof_preds[val_idx] + cat_oof_preds[val_idx]) / 3).argmax(axis=1))
    print(f"Fold {fold+1} XGB: {fold_xgb:.4f}")
    print(f"Fold {fold+1} LGB: {fold_lgb:.4f}")
    print(f"Fold {fold+1} CAT: {fold_cat:.4f}")
    print(f"Fold {fold+1} Ensemble: {fold_ensemble:.4f}")

# 전체 OOF 점수
oof_preds = (xgb_oof_preds + lgb_oof_preds + cat_oof_preds) / 3
y_encoded = y_full.values
oof_score = balanced_accuracy_score(y_encoded, oof_preds.argmax(axis=1))
print(f"\n{'='*50}")
print(f"전체 OOF 3-Model Ensemble: {oof_score:.4f}")

# 최종 test 예측 (보정 전, 참고용)
ensemble_test = (xgb_test_preds + lgb_test_preds + cat_test_preds) / 3


## 8. 분석 — OOF Confusion Matrix

보정 전 3-model ensemble의 OOF 예측으로 confusion matrix를 그려, EDA에서 예상했던 STAR↔GALAXY confusion이 실제로 나타나는지 확인합니다.

In [ ]:
oof_pred_labels = oof_preds.argmax(axis=1)
cm = confusion_matrix(y_encoded, oof_pred_labels, normalize='true')

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
disp.plot(ax=ax, cmap='Blues', values_format='.3f', colorbar=False)
ax.set_title('OOF Confusion Matrix (row-normalized)')
plt.tight_layout()
plt.show()


## 9. 분석 — Feature Importance

5-fold에 걸쳐 평균낸 LightGBM / XGBoost / CatBoost feature importance 상위 15개를 확인합니다.

> 세 모델의 importance는 계산 방식이 달라 (LightGBM: split 횟수 기준, XGBoost: gain 기준, CatBoost: PredictionValuesChange 기준) **모델 간 절대값 비교는 의미가 없고**, 각 모델 안에서의 상대적 순위만 비교하는 용도입니다.

In [ ]:
lgb_imp_mean = np.mean(lgb_importances, axis=0)
xgb_imp_mean = np.mean(xgb_importances, axis=0)
cat_imp_mean = np.mean(cat_importances, axis=0)

lgb_imp_df = pd.DataFrame({'feature': features_lgb, 'importance': lgb_imp_mean}) \
    .sort_values('importance', ascending=False).head(15)
xgb_imp_df = pd.DataFrame({'feature': features_xgb, 'importance': xgb_imp_mean}) \
    .sort_values('importance', ascending=False).head(15)
# CatBoost는 features_lgb와 동일한 feature set(X_cat_full = train[features_lgb])으로 학습됨
cat_imp_df = pd.DataFrame({'feature': features_lgb, 'importance': cat_imp_mean}) \
    .sort_values('importance', ascending=False).head(15)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
sns.barplot(data=lgb_imp_df, x='importance', y='feature', ax=axes[0], color='#4C72B0')
axes[0].set_title('LightGBM — Top 15 Feature Importance (avg over folds)')

sns.barplot(data=xgb_imp_df, x='importance', y='feature', ax=axes[1], color='#DD8452')
axes[1].set_title('XGBoost — Top 15 Feature Importance (avg over folds)')

sns.barplot(data=cat_imp_df, x='importance', y='feature', ax=axes[2], color='#55A868')
axes[2].set_title('CatBoost — Top 15 Feature Importance (avg over folds)')

plt.tight_layout()
plt.show()


## 10. Threshold 보정 (Balanced Accuracy 최적화)

Balanced Accuracy는 클래스별 recall의 평균이라 단순 argmax보다 클래스별 확률에 곱해줄 최적의 multiplier를 찾으면 점수를 더 끌어올릴 수 있습니다. `scipy.optimize.minimize`(Nelder-Mead)로 OOF 예측에 대해 이 multiplier를 탐색합니다.

In [ ]:
from scipy.optimize import minimize

print("\nRunning SciPy optimization to find perfect Balanced Accuracy thresholds...")

def balanced_objective(weights):
    adjusted_probs = oof_preds * weights
    preds = np.argmax(adjusted_probs, axis=1)
    return -balanced_accuracy_score(y_encoded, preds)

# 초기값: 클래스 빈도 역수
class_priors = np.bincount(y_encoded) / len(y_encoded)
initial_weights = 1.0 / class_priors

res = minimize(balanced_objective, initial_weights, method='Nelder-Mead')
best_weights = res.x

print(f"Optimal Multipliers Found: {best_weights}")
print(f"Calibrated 3-Model OOF Balanced Accuracy: {-res.fun:.5f}")

# 검증: 보정 전후 비교
print(f"보정 전 OOF: {oof_score:.5f}")
print(f"보정 후 OOF: {-res.fun:.5f}")
print(f"개선폭: {-res.fun - oof_score:+.5f}")


## 11. 최종 블렌딩 및 Submission 생성

In [ ]:
print("\nBlending models...")

# 앞 단계에서 fold loop 안에서 이미 누적 평균까지 끝난 상태
# xgb_test_preds, lgb_test_preds, cat_test_preds = (N_test, 3) shape

# Base 3-Way Equal Blend
final_probs = (xgb_test_preds * 0.3333) + (lgb_test_preds * 0.3334) + (cat_test_preds * 0.3333)

# 앞서 찾은 calibrated threshold multipliers 적용
calibrated_final_probs = final_probs * best_weights

final_class_indices = np.argmax(calibrated_final_probs, axis=1)
final_classes = le.inverse_transform(final_class_indices)

submission = pd.DataFrame({
    'id': test_df['id'],
    'class': final_classes
})
submission.to_csv('submission_final.csv', index=False)

print(f"Submission shape: {submission.shape}")
print(f"Predicted class distribution:\n{submission['class'].value_counts()}")


In [ ]:
submission['class'].value_counts().reindex(['GALAXY', 'QSO', 'STAR']).plot(
    kind='bar', color=['#4C72B0', '#DD8452', '#55A868'], figsize=(6, 4)
)
plt.title('Final Submission — Predicted Class Distribution')
plt.ylabel('count')
plt.tight_layout()
plt.show()


## 12. 결과 요약 & 회고

### 실제 실행 결과 (기록용)
| 단계 | Balanced Accuracy (OOF) |
|---|---|
| Base model | 0.95602 |
| + sample_weight 튜닝 | 0.96348 |
| + 5-Fold CV | 0.96438 |
| + Optuna 튜닝 + K-fold | 0.96459 |
| + 3-model ensemble | **0.96363** (보정 전) → **0.96658** (threshold 보정 후) |
| **Public LB (최종 제출)** | **0.96744** |

Fold별 개별 모델 성능은 XGB/LGB/CAT 모두 0.962~0.964 구간으로 매우 유사했고, 3-model ensemble이 개별 모델 대비 약 +0.0005~0.001 수준의 소폭 개선에 그쳤습니다.

### 배운 점 (Lessons Learned)
1. **앙상블 다양성(diversity)의 한계**: XGBoost, LightGBM, CatBoost 세 모델 모두 동일한 feature set과 거의 동일한 confusion 패턴을 학습했기 때문에, 모델 개수를 늘려도 앙상블 이득이 크지 않았습니다. 다음 대회에서는 개별 모델의 정확도를 독립적으로 최대화하기보다, **서로 다른 confusion 패턴을 갖는 모델 조합**(예: 다른 feature subset, 다른 loss function, 선형 모델 포함)을 목표로 설계할 필요가 있습니다.
2. **EDA 인사이트와 구현 사이의 갭**: EDA에서는 `spectral_type × galaxy_population` 상호작용이 STAR↔GALAXY confusion의 핵심 원인으로 확인됐지만, 실제 `sample_weight` 구현은 이 상호작용을 직접 겨냥하지 못했고 `redshift_x_uz`, `redshift_x_ug` 등 interaction feature도 QSO 축(redshift)에 집중되어 있었습니다. EDA에서 발견한 신호를 feature/전략 설계에 더 정확히 반영하는 과정이 필요합니다.
3. **공간 좌표 처리 방식 재검토**: `coord_x/y/z`를 축별 독립 quantile binning(500 bins)으로 이산화했는데, 이는 구면(sphere) 위의 점을 축별로 분리해버려 실제 "하늘에서 가까운 영역"이라는 공간적 의미가 일부 손실될 수 있습니다. 다음에는 3D 좌표에 대한 클러스터링(KMeans) 또는 천문학 표준 기법인 HEALPix를 raw 좌표 baseline과 비교해볼 계획입니다.

> House Prices 대회([thinkrunner/kaggle-house-prices](https://github.com/thinkrunner/kaggle-house-prices))에 이어, Kaggle Playground Series를 "기법 연습장(technique arsenal)"으로 삼아 다음 대회에서는 개별 모델 성능보다 앙상블 다양성 설계에 더 집중할 예정입니다.
